Experiment 1: Does our validation strategy give trustworthy estimates?

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/clean/train.csv")
df["Date"] = pd.to_datetime(df["Date"])

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 844392 entries, 0 to 844391
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Store          844392 non-null  int64         
 1   DayOfWeek      844392 non-null  int64         
 2   Date           844392 non-null  datetime64[us]
 3   Sales          844392 non-null  int64         
 4   Customers      844392 non-null  int64         
 5   Open           844392 non-null  int64         
 6   Promo          844392 non-null  int64         
 7   StateHoliday   844392 non-null  str           
 8   SchoolHoliday  844392 non-null  int64         
dtypes: datetime64[us](1), int64(7), str(1)
memory usage: 58.0 MB


In [3]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Store,844392.0,558.42292,1.0,280.0,558.0,837.0,1115.0,321.731914
DayOfWeek,844392.0,3.520361,1.0,2.0,3.0,5.0,7.0,1.723689
Date,844392,2014-04-11 01:02:42.487564,2013-01-01 00:00:00,2013-08-16 00:00:00,2014-03-31 00:00:00,2014-12-10 00:00:00,2015-07-31 00:00:00,NaN
Sales,844392.0,6955.514291,0.0,4859.0,6369.0,8360.0,41551.0,3104.21468
Customers,844392.0,762.728395,0.0,519.0,676.0,893.0,7388.0,401.227674
Open,844392.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
Promo,844392.0,0.446352,0.0,0.0,0.0,1.0,1.0,0.497114
SchoolHoliday,844392.0,0.19358,0.0,0.0,0.0,0.0,1.0,0.395103


In [4]:
test = pd.read_csv("../data/test.csv")
test = test.drop(columns=["Open"])

test["Date"] = pd.to_datetime(test["Date"])
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 41088 entries, 0 to 41087
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Id             41088 non-null  int64         
 1   Store          41088 non-null  int64         
 2   DayOfWeek      41088 non-null  int64         
 3   Date           41088 non-null  datetime64[us]
 4   Promo          41088 non-null  int64         
 5   StateHoliday   41088 non-null  str           
 6   SchoolHoliday  41088 non-null  int64         
dtypes: datetime64[us](1), int64(5), str(1)
memory usage: 2.2 MB


In [5]:
test.describe().T

,count,mean,min,25%,50%,75%,max,std
Id,41088.0,20544.5,1.0,10272.75,20544.5,30816.25,41088.0,11861.228267
Store,41088.0,555.899533,1.0,279.75,553.5,832.25,1115.0,320.274496
DayOfWeek,41088.0,3.979167,1.0,2.0,4.0,6.0,7.0,2.015481
Date,41088,2015-08-24 12:00:00,2015-08-01 00:00:00,2015-08-12 18:00:00,2015-08-24 12:00:00,2015-09-05 06:00:00,2015-09-17 00:00:00,NaN
Promo,41088.0,0.395833,0.0,0.0,0.0,1.0,1.0,0.489035
SchoolHoliday,41088.0,0.443487,0.0,0.0,0.0,1.0,1.0,0.496802


In [6]:
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [7]:
df['Open'].unique()
# drop it
df.drop(columns=["Open"], inplace=True)
df.columns

C:\Users\ecole\AppData\Local\Temp\ipykernel_66540\1618845791.py:3: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["Open"], inplace=True)


Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Promo',
       'StateHoliday', 'SchoolHoliday'],
      dtype='str')

## Forecast-origin and target-label construction.


In [8]:
df.head(3)

,Store,DayOfWeek,Date,Sales,Customers,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,0,1
1,2,5,2015-07-31,6064,625,1,0,1
2,3,5,2015-07-31,8314,821,1,0,1


In [9]:
df.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Promo',
       'StateHoliday', 'SchoolHoliday'],
      dtype='str')

In [10]:

# 1. sort bystore, and order Date chronically
df = df.sort_values(by=["Store", "Date"], ascending=[True, True])
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Promo,StateHoliday,SchoolHoliday
843264,1,3,2013-01-02,5530,668,0,0,1
842155,1,4,2013-01-03,4327,578,0,0,1
841047,1,5,2013-01-04,4486,619,0,0,1
839940,1,6,2013-01-05,4997,635,0,0,1
838815,1,1,2013-01-07,7176,785,1,0,1


In [11]:
df.tail()

,Store,DayOfWeek,Date,Sales,Customers,Promo,StateHoliday,SchoolHoliday
5564,1115,1,2015-07-27,10712,608,1,0,1
4451,1115,2,2015-07-28,8093,500,1,0,1
3338,1115,3,2015-07-29,7661,473,1,0,1
2225,1115,4,2015-07-30,8405,502,1,0,1
1112,1115,5,2015-07-31,8680,538,1,0,1


In [12]:
assert not df.duplicated(['Store', 'Date']).any()

In [13]:
# 2. generate forcast origins
# for each store, i consider each date a potential forcast origin
# only valid if it has

origin_df = df[["Store", "Date"]].copy()
origin_df = origin_df.rename(columns={"Date": "origin_date"})
origin_df.head()

,Store,origin_date
843264,1,2013-01-02
842155,1,2013-01-03
841047,1,2013-01-04
839940,1,2013-01-05
838815,1,2013-01-07


In [14]:
# 3. generate target dates for each horizon

training_rows = []
horizons = [1, 7, 14, 21, 28]

for horizon in horizons:
    temp = origin_df.copy()

    temp['forcast_horizon'] = horizon
    temp["target_date"] = temp["origin_date"] + pd.Timedelta(days=horizon)
    
    training_rows.append(temp)

forcast_df = pd.concat(training_rows, ignore_index=True)
forcast_df.head()

,Store,origin_date,forcast_horizon,target_date
0,1,2013-01-02,1,2013-01-03
1,1,2013-01-03,1,2013-01-04
2,1,2013-01-04,1,2013-01-05
3,1,2013-01-05,1,2013-01-06
4,1,2013-01-07,1,2013-01-08


In [15]:
forcast_df.loc[forcast_df['forcast_horizon'] == 7]

,Store,origin_date,forcast_horizon,target_date
844392,1,2013-01-02,7,2013-01-09
844393,1,2013-01-03,7,2013-01-10
844394,1,2013-01-04,7,2013-01-11
844395,1,2013-01-05,7,2013-01-12
844396,1,2013-01-07,7,2013-01-14
...,...,...,...,...
1688779,1115,2015-07-27,7,2015-08-03
1688780,1115,2015-07-28,7,2015-08-04
1688781,1115,2015-07-29,7,2015-08-05
1688782,1115,2015-07-30,7,2015-08-06


In [16]:
forcast_df.shape

(4221960, 4)

In [17]:
origin_df.head()

,Store,origin_date
843264,1,2013-01-02
842155,1,2013-01-03
841047,1,2013-01-04
839940,1,2013-01-05
838815,1,2013-01-07


In [18]:
origin_df.shape

(844392, 2)

horizon length=5
origion length = 844,392
training rows should be
844,392 * 5 = 4,221,960

forcast_df is correct in length
for each horizon:
    for each store forcast_horizon = horizon(e.g 1)
    repeat for each date in a store
    like store 1 across all date set forcat_horizon = horizon(e.g 1)
    when all date has finish
    duplicate store 1 across all date again this time set forcat_horizon = the next horizon(e.g 7)
    until all horizon finish


In [19]:
# 4th step to be continued...

In [20]:
forcast_df.head()

,Store,origin_date,forcast_horizon,target_date
0,1,2013-01-02,1,2013-01-03
1,1,2013-01-03,1,2013-01-04
2,1,2013-01-04,1,2013-01-05
3,1,2013-01-05,1,2013-01-06
4,1,2013-01-07,1,2013-01-08


In [21]:
forcast_df['target_date'].duplicated().any()
# True. okay fair, different stores can have same origin date thus targeting same target_date

np.True_

In [22]:
forcast_df.loc[forcast_df['target_date'].duplicated(keep=False)].sort_values('target_date')

,Store,origin_date,forcast_horizon,target_date
515314,682,2013-01-01,1,2013-01-02
400916,530,2013-01-01,1,2013-01-02
387281,512,2013-01-01,1,2013-01-02
194139,259,2013-01-01,1,2013-01-02
425029,562,2013-01-01,1,2013-01-02
...,...,...,...,...
3713731,444,2015-07-31,28,2015-08-28
4185343,1067,2015-07-31,28,2015-08-28
3712954,443,2015-07-31,28,2015-08-28
3719188,451,2015-07-31,28,2015-08-28


In [23]:
forcast_df.duplicated(subset=['Store', 'target_date']).any()

np.True_

In [24]:
forcast_df.loc[
    forcast_df.duplicated(subset=["Store", "target_date"], keep=False)
].sort_values(['Store', 'target_date']).head(15)

,Store,origin_date,forcast_horizon,target_date
5,1,2013-01-08,1,2013-01-09
844392,1,2013-01-02,7,2013-01-09
6,1,2013-01-09,1,2013-01-10
844393,1,2013-01-03,7,2013-01-10
7,1,2013-01-10,1,2013-01-11
844394,1,2013-01-04,7,2013-01-11
8,1,2013-01-11,1,2013-01-12
844395,1,2013-01-05,7,2013-01-12
10,1,2013-01-14,1,2013-01-15
844397,1,2013-01-08,7,2013-01-15


- After checking for duplicate `target_date` values, I checked for duplicates using both `Store` and `target_date`.
- Found duplicate combinations.
- Looking at the rows, the same store can have multiple forecasts targeting the same date, but generated from different origin dates and forecast horizons.
- For example, Store 1 has two forecasts targeting 2013-01-09:
  - Origin date: 2013-01-08, forecast horizon: 1
  - Origin date: 2013-01-02, forecast horizon: 7

This is expected because the same target date can be forecast from different points in time. These are different forecasting origins, not necessarily duplicate records.

A unique forcast obseration is identified by:
`Store + origin_date + forecast_horizon`

In [25]:
forcast_df.shape

(4221960, 4)

In [26]:
# 4. lookup the actual target sales
sales_lookup = df[["Store", "Date", "Sales"]].rename(
    columns={"Date": "target_date", "Sales": "target_sales"}
)

forcast_df = forcast_df.merge(
    sales_lookup,
    on=["Store", "target_date"],
    how="left",
    validate="many_to_one",  # many rows from forcast all map to 1 sales_lookup target_date
)
forcast_df.head()

,Store,origin_date,forcast_horizon,target_date,target_sales
0,1,2013-01-02,1,2013-01-03,4327.0
1,1,2013-01-03,1,2013-01-04,4486.0
2,1,2013-01-04,1,2013-01-05,4997.0
3,1,2013-01-05,1,2013-01-06,NaN
4,1,2013-01-07,1,2013-01-08,5580.0


In [27]:
df.loc[df['Store']==1].head(7)

,Store,DayOfWeek,Date,Sales,Customers,Promo,StateHoliday,SchoolHoliday
843264,1,3,2013-01-02,5530,668,0,0,1
842155,1,4,2013-01-03,4327,578,0,0,1
841047,1,5,2013-01-04,4486,619,0,0,1
839940,1,6,2013-01-05,4997,635,0,0,1
838815,1,1,2013-01-07,7176,785,1,0,1
837710,1,2,2013-01-08,5580,654,1,0,1
836605,1,3,2013-01-09,5471,626,1,0,1


In [28]:
forcast_df.isna().sum()

Store                   0
origin_date             0
forcast_horizon         0
target_date             0
target_sales       350134
dtype: int64

yes, some Store don't have a recorded sale for some target date
what to do?
i don't think i should treat it as missing value and impute with something
becasue truely there is no record for it
solution:
remove invalide forcast origin


In [29]:

forcast_df.loc[forcast_df["origin_date"] == "2013-01-05"].sort_values(
    ["Store", 'forcast_horizon']
).head(15)

,Store,origin_date,forcast_horizon,target_date,target_sales
3,1,2013-01-05,1,2013-01-06,NaN
844395,1,2013-01-05,7,2013-01-12,4952.0
1688787,1,2013-01-05,14,2013-01-19,5182.0
2533179,1,2013-01-05,21,2013-01-26,5598.0
3377571,1,2013-01-05,28,2013-02-02,5970.0
784,2,2013-01-05,1,2013-01-06,NaN
845176,2,2013-01-05,7,2013-01-12,2630.0
1689568,2,2013-01-05,14,2013-01-19,2399.0
2533960,2,2013-01-05,21,2013-01-26,2205.0
3378352,2,2013-01-05,28,2013-02-02,2956.0


- group rows with same `Store` and `origin_date`
example:
```
Store 1, 2013-01-02
Store 1, 2013-01-02	
Store 1, 2013-01-05
```

- select `target_sales` and count
- For each group, count how many non-null `target_sales` values it contains.
- `transform()` returns a result(the count of `target_sales`) for every row
- sticking to the above example:
```
Store 1, 2013-01-02, `target_sales` count = 5
Store 1, 2013-01-02, `target_sales` count = 5
Store 1, 2013-01-05, `target_sales` count = 4
```

In [30]:
# keep only forcast origins with all 5 targets observations
# becasue we want to evaluate forecasts at horizons 1, 7, 14, 21 and 28, we need complete observations for all five horizons.

valid_origins = (
    forcast_df.groupby(['Store', 'origin_date'])['target_sales']
    .transform('count') == len(horizons)
)
valid_origins   

0           True
1           True
2           True
3          False
4           True
           ...  
4221955    False
4221956    False
4221957    False
4221958    False
4221959    False
Name: target_sales, Length: 4221960, dtype: bool

In [31]:
forcast_df = forcast_df[valid_origins].copy()

forcast_df = forcast_df.sort_values(
    ['Store', 'origin_date', 'forcast_horizon']
)
forcast_df.shape

(2808265, 5)

In [32]:
forcast_df.head(25)

,Store,origin_date,forcast_horizon,target_date,target_sales
0,1,2013-01-02,1,2013-01-03,4327.0
844392,1,2013-01-02,7,2013-01-09,5471.0
1688784,1,2013-01-02,14,2013-01-16,4008.0
2533176,1,2013-01-02,21,2013-01-23,5578.0
3377568,1,2013-01-02,28,2013-01-30,4601.0
1,1,2013-01-03,1,2013-01-04,4486.0
844393,1,2013-01-03,7,2013-01-10,4892.0
1688785,1,2013-01-03,14,2013-01-17,4044.0
2533177,1,2013-01-03,21,2013-01-24,5195.0
3377569,1,2013-01-03,28,2013-01-31,4709.0


In [33]:
# df.loc[df["Date"] == "2013-01-25" and df['Store'] == 1]
df.loc[df["Date"] == "2013-01-25"]
# store 1 sals on 2013-01-04 = 5586
# correct with forcast_df row index 2533178

,Store,DayOfWeek,Date,Sales,Customers,Promo,StateHoliday,SchoolHoliday
821081,1,5,2013-01-25,5586,676,1,0,0
821082,2,5,2013-01-25,4589,522,1,0,0
821083,3,5,2013-01-25,7565,773,1,0,0
821084,4,5,2013-01-25,9730,1409,1,0,0
821085,5,5,2013-01-25,4960,564,1,0,0
...,...,...,...,...,...,...,...,...
822184,1111,5,2013-01-25,5114,485,1,0,0
822185,1112,5,2013-01-25,13661,1064,1,0,0
822186,1113,5,2013-01-25,6714,778,1,0,0
822187,1114,5,2013-01-25,19446,2838,1,0,0


In [34]:
forcast_df.isna().sum()

Store              0
origin_date        0
forcast_horizon    0
target_date        0
target_sales       0
dtype: int64

In [35]:
# forcast_df.to_csv("../data/clean/train_forcast.csv", index=False)


A forecast origin is the date at which you stand and ask:
"What will sales be in the future?"

I use dates as forcast origin provded there is enough future data for the 5 horizon.

to avoid future leakage
I use only information available at forcast origin.

Important: Historical sales features such as lags must only use sales available at the forecast origin. Target-date calendar and promotion features are allowed only when those values are genuinely known in advance.

I am imagining this business makes a new forcast everyday.

Within each store valid forcast origin = t if t+28 <= available date


validation question is:

> How should I evaluate a four-week forecasting model to establish whether its predictions will remain reliable when predicting future harvests?

I decided to evaluate at 1, 7, 14, 21 and 28 days ahead.

To investigate how forecast accuracy changes with prediction horizon.

| Horizon | What you're evaluating         |
| ------- | ------------------------------ |
| 1 day   | Predicting tomorrow's sales    |
| 7 days  | Predicting sales 7 days ahead  |
| 14 days | Predicting sales 14 days ahead |
| 21 days | Predicting sales 21 days ahead |
| 28 days | Predicting sales 28 days ahead |



# Split Dataset

## 1. When to split the dataset

I will split the dataset before feature engineering, partly because I am considering using lag and rolling-window features.

I started by listing the available validation strategies I know and reasoning through which ones are appropriate for this experiment.

The split should not randomly shuffle or select data points. It should preserve the chronological order of the data.

I will therefore arrange the dataset chronologically before splitting it.

## 2. Validation strategies considered

The validation strategies I considered are:

1. Hold-out train-validation split
2. K-Fold cross-validation
   - Group K-Fold (`group=Store`)
   - Expanding-window (Walk-Forward) CV
   - Rolling-window CV
3. Out-of-Time (OOT) validation
   - Static holdout

### Why I ruled out some options

**Hold-out train-validation split**

This performs only one split. The model is evaluated on one validation period, so the result could be influenced by how that particular period behaves.

I want to evaluate the model across multiple future periods to understand how consistently it performs.

**Static OOT holdout**

This trains on a fixed historical period and evaluates on a later period. The split does not move.

It is similar to a chronological train-validation split, so using it alone would give me only one validation period.

I will still reserve a separate static OOT period for final evaluation, but I will not use it as my only validation strategy.

**Clarifying OOT validation**

OOT validation means evaluating a model on data that comes after its training data.

A static holdout, expanding-window CV, and rolling-window CV can all follow this principle. OOT is not a separate method that replaces expanding or rolling CV.

## 3. How I will split the data

My experiment asks:

> How should I evaluate a four-week forecasting model to establish whether its predictions will remain reliable when predicting future harvests?

I will use multiple temporal validation folds and reserve a final untouched static OOT test period.

I will first reserve the last 3 months of the training dataset for static OOT evaluation.

The remaining data will be used for temporal cross-validation.

## 4. Cross-validation strategies

### Group K-Fold

Group K-Fold tests whether the model can generalize to stores it has not seen during training.

That is not what this experiment is asking.

I already have historical data for the stores. The question is:

Can the model predict future sales accurately for stores whose historical data it has already observed?

I can include `Store` as a feature so the model can learn store-specific patterns.

### Expanding-window (Walk-Forward) CV

This tests whether incorporating more historical observations improves future predictions.

The training window grows with each fold. Earlier observations remain in the training data as newer observations are added.

### Rolling-window CV

This tests whether more recent observations are sufficient to produce reliable predictions.

The training window has a fixed size and moves forward with each fold. Older observations are dropped as newer observations are added.

## 5. What kind of generalization do I want to test?

I want to test **temporal generalization**.

The model already has historical observations from the stores. I want to understand whether it can use that historical information to make reliable predictions for future dates.

I also want to understand whether using more historical data or focusing on more recent data produces better results.

## 6. Validation rules and important considerations

**Evaluate all five forecast horizons in every fold.**

Each validation fold should evaluate:

- 1 day ahead
- 7 days ahead
- 14 days ahead
- 21 days ahead
- 28 days ahead

I will calculate performance separately for each horizon to understand how prediction accuracy changes as the forecast horizon increases.

**Training cutoff must respect both `origin_date` and `target_date`.**

At each fold's training cutoff, only training examples whose `target_date` is on or before that cutoff can be used.

Otherwise, the model would learn from outcomes that had not happened yet.

For example:

```python
train_df = train_df[
    train_df["target_date"] <= training_cutoff
]
```

For rolling-window CV, I must also restrict the training data to the selected rolling window. For expanding-window CV, I retain the earlier training history.

This is important because I cannot train a model on labels that were not available at the training cutoff.

- Training: target labels must be available by the training cutoff.
- Validation: target labels must exist in our historical dataset so we can compare predictions against actual sales later. They must be future outcomes relative to the forecast origin.

The validation target values are for evaluation, not model fitting.

**Initial training window**

I will start with a 12-month initial training window.

This gives the model an opportunity to observe a complete annual cycle before its first validation period.

**Overlapping forecast targets**

Since my forecasting origins are daily, different origins can have overlapping forecast targets.

For example, the 28-day forecast from one origin may target the same date as the 27-day forecast from the following day's origin.

This is not automatically leakage. It reflects a business issuing forecasts daily.

However, these observations are correlated, so I should keep that in mind when interpreting evaluation results.

In [36]:

date = pd.Timestamp("2014-03-02")
new_date = date + pd.DateOffset(days=28)


print(new_date)
# 2013-02-27


2014-03-30 00:00:00


In [37]:
# 1. remove last 3 month untouchedfor test

test_start = forcast_df["origin_date"].max() - pd.DateOffset(months=3)

test_forcast_df = forcast_df.loc[forcast_df["origin_date"] >= test_start].copy()
train_forcast_df = forcast_df.loc[forcast_df["origin_date"] < test_start].copy()

In [38]:
test_start

Timestamp('2015-04-03 00:00:00')

In [39]:
# test_forcast_df origin_date
# min: 2015-04-03
# max: 2015-07-03

# train_forcast_df origin_date
# min: 2013-01-01 max: 2015-04-02

## Expanding window

In [40]:
# actual splitting

from sklearn.model_selection import TimeSeriesSplit


In [41]:
# Number of days in the initial training window and validation block
initial_train_days = 365
validation_days = 28

# Get unique forecast origin dates, in chronological order
origin_dates = origin_df['origin_date'].unique()

# Calculate how many folds we can create while keeping
# at least 365 days in the first training window
n_splits = (len(origin_dates) - initial_train_days) // validation_days

# Expanding-window split
tscv = TimeSeriesSplit(n_splits=n_splits, test_size=validation_days, max_train_size=None)

In [42]:
n_splits

20

In [43]:
# Generate the training and validation data for each fold
folds = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X=origin_dates), start=1):

    # get the dates assigned to training and validation
    train_dates = origin_dates[train_idx]
    val_dates = origin_dates[val_idx]

    # select all rows belonging to those dates
    fold_train = train_forcast_df[
        train_forcast_df['origin_date'].isin(train_dates)
    ].copy()

    fold_val = train_forcast_df[
        train_forcast_df['origin_date'].isin(val_dates)
    ].copy()

    # Only use training labels whose outcomes were available
    # by the beginning of the validation period
    # substrct 1 day from cutoff because current day sales hasn't happen yet
    training_cutoff = val_dates[0] - pd.Timedelta(days=1)

    fold_train = fold_train[
        fold_train['target_date'] <= training_cutoff
    ].copy()

    folds.append({
        'fold': fold,
        'train': fold_train,
        'validation': fold_val,
        'training_cutoff': training_cutoff
    })

In [44]:
# Inspect folds

for item in folds:
    print(f"Fold {item['fold']}")

    print(
        "Training origins:",
        item["train"]["origin_date"].min(),
        "to",
        item["train"]["origin_date"].max(),
    )

    print("Training target cutoff:", item["training_cutoff"])

    print(
        "Validation origins:",
        item["validation"]["origin_date"].min(),
        "to",
        item["validation"]["origin_date"].max(),
    )

    print()

Fold 1
Training origins: 2013-01-02 00:00:00 to 2014-04-02 00:00:00
Training target cutoff: 2014-04-03 00:00:00
Validation origins: 2014-04-04 00:00:00 to 2014-05-09 00:00:00

Fold 2
Training origins: 2013-01-02 00:00:00 to 2014-05-08 00:00:00
Training target cutoff: 2014-05-09 00:00:00
Validation origins: 2014-05-10 00:00:00 to 2014-06-13 00:00:00

Fold 3
Training origins: 2013-01-02 00:00:00 to 2014-06-12 00:00:00
Training target cutoff: 2014-06-13 00:00:00
Validation origins: 2014-06-14 00:00:00 to 2014-07-17 00:00:00

Fold 4
Training origins: 2013-01-02 00:00:00 to 2014-07-16 00:00:00
Training target cutoff: 2014-07-17 00:00:00
Validation origins: 2014-07-18 00:00:00 to 2014-08-19 00:00:00

Fold 5
Training origins: 2013-01-02 00:00:00 to 2014-08-18 00:00:00
Training target cutoff: 2014-08-19 00:00:00
Validation origins: 2014-08-20 00:00:00 to 2014-09-20 00:00:00

Fold 6
Training origins: 2013-01-02 00:00:00 to 2014-09-20 00:00:00
Training target cutoff: 2014-09-21 00:00:00
Validati

In [45]:
print("Total unique origin dates:", len(origin_dates))
print("Number of folds:", n_splits)

# Inspect the first fold before filtering by target_date
train_idx, val_idx = next(tscv.split(origin_dates))

print("Training observations:", len(train_idx))
print("Validation observations:", len(val_idx))

print(
    "Training calendar span:",
    origin_dates[train_idx[0]],
    "to",
    origin_dates[train_idx[-1]],
)

print(
    "Validation calendar span:",
    origin_dates[val_idx[0]],
    "to",
    origin_dates[val_idx[-1]],
)


Total unique origin dates: 942
Number of folds: 20
Training observations: 382
Validation observations: 28
Training calendar span: 2013-01-02 00:00:00 to 2014-04-03 00:00:00
Validation calendar span: 2014-04-04 00:00:00 to 2014-05-09 00:00:00


Example:
- April 4 is the forecast day.
- We stand on April 4 and generate a forecast.
- April 4 sales haven't happened yet.
- April 3 sales have already been recorded.
- Thus I am allowing training examples whose target dates are up to April 3.
    training cutoff = April 3

Training cutoff determines which target_sales values the model is allowed to learn from.

- I use validation date to determine training cutoff
because validation date tells me when my first forcast will happen
And that tells me how much historical infromation is available at that time.
- April 4 is my first forcast date
  April 4 sales hasn't happened yet
  I can use historical infromation whose target date is up to April 3


TimeSeriesSplit setup:
Your origin_dates probably doesn't contain every calendar date.
so available dates not consecutive dates.

- Validation count: Equal across all folds (28 unique origin-date observations each).
- Training count: Increases across folds because you're using expanding-window CV (max_train_size=None).

One important detail: after applying `target_date <= training_cutoff` filter, the final number of training rows can vary depending on how many examples get excluded.

- Validation block is 28 days for each fold,
because I want my model to be evaluated over that period before next training.

## Rolling window TimeSeriesSplit

In [ ]:
# Calculate how many folds we can create while keeping
# at least 365 days in the first training window
n_splits = (len(origin_dates) - initial_train_days) // validation_days

# Rolling window
# train size remains 12 months, validation size is 1 month
rolling_tscv = TimeSeriesSplit(n_splits=n_splits, max_train_size=initial_train_days, test_size=validation_days)

In [ ]:
# Generate the training and validation data for each fold
rolling_folds = []

for fold, (train_idx, val_idx) in enumerate(
    rolling_tscv.split(X=origin_dates), start=1
):
    # Get the dates assigned to training and validation
    train_dates = origin_dates[train_idx]
    val_dates = origin_dates[val_idx]

    # Select all rows belonging to those dates
    fold_train = train_forcast_df[
        train_forcast_df['origin_date'].isin(train_dates)
    ].copy()

    fold_val = train_forcast_df[
        train_forcast_df['origin_date'].isin(val_dates)
    ].copy()

    # Only use training labels whose outcomes were available
    # by the beginning of the validation period
    training_cutoff = val_dates[0] - pd.Timedelta(days=1)
    fold_train = fold_train[
        fold_train['target_date'] <= training_cutoff
    ].copy()

    rolling_folds.append(
        {
            "fold": fold,
            "train": fold_train,
            "validation": fold_val,
            "training_cutoff": training_cutoff,
        }
    )

In [ ]:
# Inspect the folds
for item in rolling_folds:
    print(f"Fold {item['fold']}")

    print(
        "Training origins:",
        item["train"]["origin_date"].min(),
        "to",
        item["train"]["origin_date"].max(),
    )

    print("Training target cutoff:", item["training_cutoff"])

    print(
        "Validation origins:",
        item["validation"]["origin_date"].min(),
        "to",
        item["validation"]["origin_date"].max(),
    )

    print()

Fold 1
Training origins: 2013-01-22 00:00:00 to 2014-04-02 00:00:00
Training target cutoff: 2014-04-03 00:00:00
Validation origins: 2014-04-04 00:00:00 to 2014-05-09 00:00:00

Fold 2
Training origins: 2013-02-23 00:00:00 to 2014-05-08 00:00:00
Training target cutoff: 2014-05-09 00:00:00
Validation origins: 2014-05-10 00:00:00 to 2014-06-13 00:00:00

Fold 3
Training origins: 2013-03-28 00:00:00 to 2014-06-12 00:00:00
Training target cutoff: 2014-06-13 00:00:00
Validation origins: 2014-06-14 00:00:00 to 2014-07-17 00:00:00

Fold 4
Training origins: 2013-05-03 00:00:00 to 2014-07-16 00:00:00
Training target cutoff: 2014-07-17 00:00:00
Validation origins: 2014-07-18 00:00:00 to 2014-08-19 00:00:00

Fold 5
Training origins: 2013-06-08 00:00:00 to 2014-08-18 00:00:00
Training target cutoff: 2014-08-19 00:00:00
Validation origins: 2014-08-20 00:00:00 to 2014-09-20 00:00:00

Fold 6
Training origins: 2013-07-11 00:00:00 to 2014-09-20 00:00:00
Training target cutoff: 2014-09-21 00:00:00
Validati

## Error metrics

- wrong sales forcast leads to wasted resources. 
- Overforecasting can lead to excess inventory, waste, or unnecessary resource allocation.
- Underforecasting can lead to stockouts, missed sales, or lost revenue.
- which is worse depends on the business
what they want to protize, and the acceptable tradeoff
so for this project I would use WAPE (Weighted Absolute Percentage Error, also known as wMAPE)
Because it calculates the total prediction error relative to the actual total sales.
Also becasue it penalizes large wrong predictions.
Tradeoff:
- It can hide poor performance on smaller sales observation
    So smaller stores with smaller sales volume, error from them would not inflence WAPE much.
- It treats overforcasting and underforcasting the same
    overestimated forcast error, and underestimated forcast error contribute the same to WAPE.
    So WAPE measures the size of the error, not its business direction.



## What counts as a **meaningful difference** between the 2 validation strategy?
- A meaningful difference is one large enough to matter for the business.
- I want to decide this now before seeing the actual error to avoid cherry picking

My comparison rules:
1. Primary comparison: Difference in WAPE between expanding and rolling windows.
2. Compare each horizon separately: 1, 7, 14, 21 and 28 days.
3. Check consistency across folds: Does rolling consistently outperform expanding, or does it only win in one fold?
4. Check forecast bias: Does either strategy systematically overforecast or underforecast?
5. Consider the training cost: If both strategies perform similarly, is the simpler strategy preferable?

For the numerical comparison, calculate:

1. Change in WAPE 
\[
\Delta WAPE = WAPE_{rolling}-WAPE_{expanding}
\]

A negative result means rolling performed better. A positive result means expanding performed better.

2. relative improvement:
\[
Relative\ Improvement =
\frac{WAPE_{expanding}-WAPE_{rolling}}
{WAPE_{expanding}}\times100
\]

This tells me the percentage improvement relative to the expanding-window result.

- I wouldn't invent a threshold such as 1% or 2% just to have one.

- Ideally, the business tells us how much forecasting improvement is worth the additional cost or complexity.

- Since I don't have that information, I'll report the measured differences, their consistency across folds, and whether the difference appears practically important. 
I won't claim that one strategy is meaningfully better just because its WAPE is slightly lower.

**CAUTION:** because my forecast origins are daily, many validation predictions will have overlapping target dates. Their errors aren't independent observations. So I shouldn't treat every prediction as an independent sample when making statistical significance claims.

- Current business objective prioritize minimizing total sales forecasting error across all stores.
<!-- I can try next: ensuring that individual stores receive accurate forecasts regardless of their sales volume? That distinction affects how we should aggregate WAPE. -->

## Feature Engineering

after splitting

making the data model ready
1. Date column
    - extract: year, month
    - drop Date
2. StateHoliday column
    - one hot encode since only 4 unique values['0', 'a', 'b', 'c'] so 3 new columns, dropping 1('0') if it's not a or b or c it's 0
3. Customers colums
    - it represents the number of customers on a given day
    - prediction is produced before the day happens
    - drop becos wouldn't be available

feature engineering:
- what to create?